# Chapter 2 v2 — NB 03: two lists for the PMBB curators (resolving the crosswalk gap)

**Goal:** produce the actionable material to request the **GENO_ID ↔ PT_ID master crosswalk** from the PMBB
curators (Nikki). The 4→8 carrier-case gap (H1) hinges on a linkage gap that has **two sides**:

- **List A — 7 WES carriers without phenotype.** They carry a rare qualifying pLOF ZNF175 variant (in the WES `.fam`),
  but their `GENO_ID` is absent from the v1 `Demographics` crosswalk → **tinnitus status unknown**.
- **List B — 77 tinnitus cases without genotype.** They meet the tinnitus rule-of-2 in the diagnoses, but have **no
  GENO_ID** → **ZNF175 carrier status unknown** (would need WES).

A carrier who is also a tinnitus case would show up on List A (as a carrier) and, if their phenotype exists but is
unlinked, on List B (as a tinnitus case with no genotype). The crosswalk is what lets us cross the two.
See `strategy_8_vs_4_carriers.md`, memory `project_upenn_pmbbid_no_bridge`.

In [1]:
import subprocess
from pathlib import Path
from collections import defaultdict
import pandas as pd

BASE = Path("/project/hall/analysis/hearing-loss-genomics")
R2   = BASE/"analysis/chapter_2_v2/results"; R2.mkdir(parents=True, exist_ok=True)
FZ   = Path("/static/PMBB/PMBB_Freeze17")
DEMO = FZ/"phenotype/PMBB_Geno_Demographics_Deidentified_012020.csv"
DIAG = FZ/"phenotype/PMBB_Geno_Nonsensitive_Diagnosis_Deidentified_012020.csv"
FAM  = FZ/"genotype/exome/all_variants/UPENN_Freeze_One_GRCh38.GL.pVCF.biallelic.fam"
BCF  = "/appl/bcftools-1.21/bin/bcftools"
VZ   = BASE/"analysis/chapter_2/results/02/v1_znf175_strict_region.vcf.gz"

demo = pd.read_csv(DEMO, dtype=str)
fam  = pd.read_csv(FAM, sep=r"\s+", header=None, usecols=[1], names=["GENO_ID"])
print("ready | Demographics rows", len(demo), "| WES .fam samples", len(fam))

ready | Demographics rows 21367 | WES .fam samples 11451


## List A — the 7 WES ZNF175 carriers with no phenotype linkage
Each row: the carrier's `GENO_ID` and the qualifying rare pLOF ZNF175 variant they carry (coord, consequence, cohort MAF).

In [2]:
carr = pd.read_csv(BASE/"analysis/chapter_2/results/06/carriers_v1.csv")
carr = carr[carr.carrier==1].copy()
linked_geno = set(demo.dropna(subset=["GENO_ID"]).GENO_ID)
carr["linked"] = carr.IID.isin(linked_geno)
unlinked = set(carr[~carr.linked].IID)
print(f"qualifying carriers: {len(carr)} | linked: {int(carr.linked.sum())} | UNLINKED: {len(unlinked)}")

# qualifying pLOF variant map (VEP)
qv = pd.read_csv(BASE/"analysis/chapter_2/results/06/znf175_qualified_variants.csv", dtype=str)
qv["vid"] = qv.CHROM+":"+qv.POS+":"+qv.REF+":"+qv.ALT
plof = set(qv[qv.is_pLOF=="True"].vid); cons = dict(zip(qv.vid, qv.Consequence))

# which qualifying variant each unlinked carrier holds + cohort MAF
q = subprocess.run([BCF,"query","-f","[%SAMPLE\t%CHROM:%POS:%REF:%ALT\t%GT\n]",str(VZ)], capture_output=True, text=True).stdout
ac=defaultdict(int); an=defaultdict(int); hold=defaultdict(list)
for ln in q.split("\n"):
    if not ln: continue
    s,vid,gt=ln.split("\t"); a=gt.count("1"); an[vid]+=gt.count("0")+a; ac[vid]+=a
    if a>0 and s in unlinked and vid in plof: hold[s].append(vid)
maf={v:(min(ac[v]/an[v],1-ac[v]/an[v]) if an[v] else 0) for v in an}

rowsA=[]
for gid in sorted(unlinked):
    for vid in hold.get(gid,[]):
        c,p,r,al = vid.split(":")
        rowsA.append({"GENO_ID":gid,"chrom":c,"pos":p,"ref":r,"alt":al,
                      "consequence":cons.get(vid,"?"),"cohort_MAF":round(maf[vid],6)})
listA = pd.DataFrame(rowsA)
listA.to_csv(R2/"nb03_list_A_wes_carriers_no_phenotype.csv", index=False)
print(listA.to_string(index=False))

qualifying carriers: 34 | linked: 27 | UNLINKED: 7


                                                 GENO_ID chrom      pos ref alt        consequence  cohort_MAF
                            UPENN_UPENN10004157_6774b761    19 51588382   C   G        stop_gained    0.000044
UPENN_UPENN10011369_bd800d5d-7a9f-4c4c-9471-81ded4f10d47    19 51581437   A  AG frameshift_variant    0.000611
UPENN_UPENN10013315_ebe556f2-2e10-4f13-8621-7384edaf41aa    19 51581437   A  AG frameshift_variant    0.000611
                               UPENN_UPENN19843_621dbffb    19 51587814   C   T        stop_gained    0.000218
                               UPENN_UPENN21914_9a42d733    19 51581437   A  AG frameshift_variant    0.000611
    UPENN_UPENN5420_29633883-8efd-4a48-9273-77fcce01efa6    19 51587814   C   T        stop_gained    0.000218
                                UPENN_UPENN8592_81e66c94    19 51588428  CA   C frameshift_variant    0.000306


## List B — the 77 tinnitus cases with no GENO_ID
Tinnitus = ICD-9 388.3x / ICD-10 H93.1x on ≥2 distinct dates (rule-of-2). Kept: those whose `PT_ID` has **no GENO_ID**
in `Demographics`. Each row carries the tinnitus evidence (n distinct dates, code span) so a curator can locate the record.

In [3]:
diag = pd.read_csv(DIAG, header=None, usecols=[0,1,2,6], names=["PT_ID","CODE","VER","DATE"], dtype=str)
tin = diag[diag.CODE.fillna("").str.match(r"^(388\.3|H93\.1)")].copy()
ev = tin.groupby("PT_ID").agg(tin_n_dates=("DATE","nunique"),
                              tin_codes=("CODE", lambda s: ",".join(sorted(set(s)))),
                              first_date=("DATE","min"), last_date=("DATE","max")).reset_index()
tin_cases = ev[ev.tin_n_dates>=2].copy()                       # rule-of-2

pt_with_geno = set(demo.dropna(subset=["GENO_ID"]).PT_ID)      # PT_IDs that have any GENO_ID
listB = tin_cases[~tin_cases.PT_ID.isin(pt_with_geno)].copy()  # tinnitus, NO genotype
# attach whatever demographics exist (sex/birth year) for these PT_IDs
dmeta = demo[["PT_ID","GENDER_CODE","BIRTH_YEAR"]].drop_duplicates("PT_ID")
listB = listB.merge(dmeta, on="PT_ID", how="left")
listB.to_csv(R2/"nb03_list_B_tinnitus_no_genotype.csv", index=False)
print(f"tinnitus cases (rule-of-2): {len(tin_cases)} | of which NO GENO_ID (List B): {len(listB)}")
print(listB.head(10).to_string(index=False))

tinnitus cases (rule-of-2): 287 | of which NO GENO_ID (List B): 77
      PT_ID  tin_n_dates     tin_codes first_date  last_date GENDER_CODE BIRTH_YEAR
10285733042            2        388.30 2015-08-12 2015-08-17           F       1938
10286172037            2 H93.11,H93.19 2018-01-28 2018-02-14           F       1956
10852783073            4        388.30 2015-01-09 2015-04-17           F       1964
10855842948            4 388.30,H93.12 2013-10-05 2017-12-16           F       1952
10859964882            2        H93.12 2019-11-02 2019-12-28           M       1958
10861674672            2        H93.12 2017-08-07 2017-11-18           M       1956
 1285764363            2 388.30,H93.13 2015-05-11 2017-10-13           F       1954
 1285910964            2        H93.12 2017-07-16 2017-09-10           F       1937
 1285926657            2        388.30 2009-07-06 2013-12-31           F       1937
 1286031537            3        388.30 2012-03-15 2012-04-28           M       1931


## Summary + framing for the curator request

In [4]:
req = f'''# PMBB curator request — ZNF175 crosswalk gap (v1 / Freeze One)

We are replicating Park et al. 2021 (ZNF175 pLOF → tinnitus) in Freeze One WES. Our qualifying-carrier × tinnitus
count is **4 carrier-cases**; a project-internal number is **8**. We have traced the gap to a **GENO_ID ↔ PT_ID
linkage gap** and need the master crosswalk to close it. Two lists (attached):

## List A — {len(listA['GENO_ID'].unique())} WES carriers with NO phenotype linkage
GENO_IDs present in the WES `.fam` carrying a rare qualifying pLOF ZNF175 variant, but ABSENT from
`PMBB_Geno_Demographics_Deidentified_012020.csv` (no PT_ID → tinnitus status unknown).
File: `nb03_list_A_wes_carriers_no_phenotype.csv`.
Ask: the PT_ID for each, or confirmation they are unconsented / QC-dropped.

## List B — {len(listB)} tinnitus cases with NO genotype
PT_IDs meeting tinnitus rule-of-2 (ICD 388.3x / H93.1x, ≥2 dates) with NO GENO_ID in Demographics
(ZNF175 carrier status untestable — would need WES).
File: `nb03_list_B_tinnitus_no_genotype.csv`.
Ask: whether any of these have WES (a GENO_ID), i.e. the reverse crosswalk.

## Why it matters
A carrier who is also a tinnitus case would sit on List A (carrier) and, if unlinked, effectively on List B
(tinnitus, no linked genotype). The crosswalk lets us cross the two and settle 4 vs 8.
Note (expectation): the linked carrier tinnitus rate is 4/27 ≈ 15%, so recovering all 7 is expected to add ~1 case
(→5), not 4 — reaching 8 would require strong enrichment among the unlinked.
'''
(R2/"nb03_curator_request.md").write_text(req)
print(req)

# PMBB curator request — ZNF175 crosswalk gap (v1 / Freeze One)

We are replicating Park et al. 2021 (ZNF175 pLOF → tinnitus) in Freeze One WES. Our qualifying-carrier × tinnitus
count is **4 carrier-cases**; a project-internal number is **8**. We have traced the gap to a **GENO_ID ↔ PT_ID
linkage gap** and need the master crosswalk to close it. Two lists (attached):

## List A — 7 WES carriers with NO phenotype linkage
GENO_IDs present in the WES `.fam` carrying a rare qualifying pLOF ZNF175 variant, but ABSENT from
`PMBB_Geno_Demographics_Deidentified_012020.csv` (no PT_ID → tinnitus status unknown).
File: `nb03_list_A_wes_carriers_no_phenotype.csv`.
Ask: the PT_ID for each, or confirmation they are unconsented / QC-dropped.

## List B — 77 tinnitus cases with NO genotype
PT_IDs meeting tinnitus rule-of-2 (ICD 388.3x / H93.1x, ≥2 dates) with NO GENO_ID in Demographics
(ZNF175 carrier status untestable — would need WES).
File: `nb03_list_B_tinnitus_no_genotype.csv`.
Ask: whether any o